## Setup

Run this notebook from a local clone of the [svm-gmu](https://github.com/SushrutGaikwad/svm-gmu) repository. From the repo root, install the project and launch Jupyter with [uv](https://docs.astral.sh/uv/):

```bash
uv sync
uv run jupyter lab
```

The next cell puts the repo's `src/` and `experiments/` folders on the import path, so `svm_gmu` and the shared helper modules (`_common` and the `*_figs` files) import from your local clone.

In [ ]:
import sys
from pathlib import Path


def _add_repo_to_path():
    """Put the repo's experiments/ and src/ folders on sys.path.

    Locates the svm-gmu repository by searching upward from the working
    directory, so the notebook runs from any local clone regardless of where
    Jupyter was started.
    """
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "experiments" / "_common.py").exists():
            exp, src = base / "experiments", base / "src"
        elif (base / "_common.py").exists():
            exp, src = base, base.parent / "src"
        else:
            continue
        for path in (str(exp), str(src)):
            if path not in sys.path:
                sys.path.insert(0, path)
        return
    raise RuntimeError(
        "Could not find the svm-gmu repository. Run this notebook from a local "
        "clone (for example, `uv run jupyter lab` from the repo root)."
    )


_add_repo_to_path()

import _common as C
import high_dim_scaling_figs as H

import matplotlib.pyplot as plt

# High-Dimensional Scaling of Sampling vs Closed-Form SVM-GMU

This experiment asks: as the feature dimension $d$ grows, how many Monte Carlo samples per training example does a standard SVM need to match the closed-form SVM-GMU boundary?

**Setup.** For each target dimension $d$ we build a *purpose-built, genuinely $d$-dimensional* dataset (see `C.make_highdim_gmm_dataset`): two close classes in $\mathbb{R}^d$, where each of the $n=10$ examples is a genuine 3-component isotropic Gaussian mixture, $f(x) = \sum_m \tfrac{1}{3}\,\mathcal{N}(x;\, c_i + s\,u_i^{(m)},\, \sigma^2 I_d)$, whose components fill all $d$ coordinates. Because every example is a real mixture ($M=3>1$), the closed-form reference is literally SVM-GMU. (This replaces an earlier construction that lifted the 2D `close_separable` data with nuisance-noise dimensions; that version kept the signal in a 2D subspace, so it was replaced by a genuinely high-dimensional one.)

**Headline question: $N^*(d)$.** For each $d$ we sweep a ladder of sample counts $N$ (samples per example), train a standard SVM on the resulting $10N$-point cloud, and record how close its boundary gets to the closed-form SVM-GMU boundary (angle in degrees). $N^*(d)$ is the smallest $N$ at which the angle falls below a threshold $\tau$. If sampling is harder in higher dimensions, $N^*(d)$ should grow with $d$.

**Secondary view: fixed-budget angle vs $d$.** For two fixed budgets $N \in \{1000, 10000\}$ we plot the median angle as a function of $d$. If higher dimensions demand more samples, the angle should increase with $d$ for any fixed $N$.

The full-resolution PGF figures for the paper (30 seeds, ladder to $3\times10^4$) are produced by `experiments/high_dim_scaling_figs.py`; this notebook runs a smoke-sized sweep for quick illustration.

In [ ]:
# Smoke-sized sweep: D_VALUES_SMOKE=[2,3,5], N_LADDER_SMOKE=[10,100,1000], R_SEEDS_SMOKE=3.
seeds = C.make_seeds(C.MASTER_SEED, H.R_SEEDS_SMOKE)
res = H.run_highdim_sweep(H.D_VALUES_SMOKE, H.N_LADDER_SMOKE, seeds)
H.make_nstar_figure(res)
plt.show()

In [ ]:
H.make_fixedbudget_figure(res)
plt.show()